# Gemini API Authentication Smoke Test

**Purpose:**
This notebook verifies that the local Gemini API key can authenticate and
complete one harmless API request.

**Scope:**
- No municipal zoning content is sent.
- No PDFs, source packets, manifests, or pilot outputs are read.
- No triage, extraction, scoring, or review workflow is run.
- This notebook is not part of the formal Stage 0–2 pilot sequence.


In [1]:
# --- Environment and safe key-presence check ---
# This cell never prints the key itself, a prefix/suffix of it, or any environment dump.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / ".env").exists() and (REPO_ROOT.parent / ".env").exists():
    REPO_ROOT = REPO_ROOT.parent  # notebook may be launched from zoning_pdf_pilot/ itself

ENV_PATH = REPO_ROOT / ".env"
env_path_exists = ENV_PATH.exists()
if env_path_exists:
    load_dotenv(ENV_PATH)

api_key = os.environ.get("GEMINI_API_KEY")
key_present = bool(api_key)
key_length = len(api_key) if api_key else 0

try:
    import google.genai as genai
    genai_version = getattr(genai, "__version__", "unknown")
except ImportError:
    genai_version = "not_installed"

print(f".env path exists: {env_path_exists}")
print(f"GEMINI_API_KEY present: {key_present}")
print(f"GEMINI_API_KEY length: {key_length}")
print(f"Active Python executable: {sys.executable}")
print(f"google-genai version: {genai_version}")

if not key_present:
    print()
    print("GEMINI_API_KEY is not present. Add it to the local .env file and rerun.")


.env path exists: True
GEMINI_API_KEY present: True
GEMINI_API_KEY length: 53
Active Python executable: /Users/sierramendoza/Documents/HAP-Repository/.venv/bin/python
google-genai version: 2.27.0


In [2]:
# --- Single harmless API request ---
# No grounding, tools, file upload, structured output, or system prompt. No zoning
# content. Exactly one request, a fixed harmless prompt, a bounded printed response.

MODEL_NAME = "gemini-flash-latest"
HARMLESS_PROMPT = "Reply with exactly one word: OK"

print("=" * 72)
print("GEMINI AUTH SMOKE TEST")
print("=" * 72)
print(f"Key present: {key_present}")
print(f"Key length: {key_length}")
print(f"Model requested: {MODEL_NAME}")

if not key_present:
    print("Request status: SKIPPED (no key present)")
else:
    from google import genai

    try:
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(model=MODEL_NAME, contents=HARMLESS_PROMPT)
        bounded_response = (response.text or "")[:200]
        print("Request status: SUCCESS")
        print(f"Response: {bounded_response}")
    except Exception as exc:
        bounded_message = str(exc)[:500]
        print("Request status: FAILED")
        print(f"Exception class: {type(exc).__name__}")
        print(f"Message (truncated to 500 chars): {bounded_message}")

print("=" * 72)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GEMINI AUTH SMOKE TEST
Key present: True
Key length: 53
Model requested: gemini-flash-latest


Request status: FAILED
Exception class: ServerError
Message (truncated to 500 chars): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


---

A successful smoke test only verifies local authentication and basic API
connectivity. It does not validate model quality, triage prompts, extraction
logic, source-packet handling, citation quality, or any zoning conclusion.

No pilot packet should be sent to Gemini until the Stage 3 triage design has
been reviewed and explicitly approved.
